In [ ]:
import pandas as pd
from typing import Tuple
import math
import numpy as np
from tqdm import tqdm
from pathlib import Path
from utils import DFPreprocessor

# Cleaning Schedule
This notebook provides a clean version of the raw given schedule {schedule_name}.csv file.
If does the following:
1. Rename duplicate columns.
2. Renaming any instances of 'ATH' to 'OAK', i.e. rebranding the Sacramento Athletics to Oakland Athletics.
3. Adding a 'timestamp' column, which provides a `pd.Timestamp` for the game start.
4. Adding 'homedistancetraveled' and 'visdistancetraveled' columns, which contains the distance in miles traveled from the team's previous games.
5. Adding 'homerestdays' and 'visrestdays' columns, which contains the number of days between the current game and the previous game for the home and visiting teams.
6. Save to a new csv, `.data/clean/{schedule_name}_cleaned.csv`

In [5]:
pd.set_option('display.max_columns', None)

In [ ]:
schedule_name = "2025schedule"

schedule = pd.read_csv(Path.cwd() / "analysis" / "data" / "raw" / f"{schedule_name}.csv")
# Add Preprocessor
preprocessor = DFPreprocessor(schedule)

schedule.head()

,Date,Num,Day,Visitor,League,Game,Home,League.1,Game.1,Day/Night,Location,Postponed,Makeup
0,20250318,0,Tuesday,LAN,NL,1,CHN,NL,1,n,TOK01,NaN,NaN
1,20250319,0,Wednesday,LAN,NL,2,CHN,NL,2,n,TOK01,NaN,NaN
2,20250327,0,Thursday,MIL,NL,1,NYA,AL,1,d,NYC21,NaN,NaN
3,20250327,0,Thursday,BAL,AL,1,TOR,AL,1,d,TOR02,NaN,NaN
4,20250327,0,Thursday,BOS,AL,1,TEX,AL,1,d,ARL03,NaN,NaN


## 1. Rename Duplicate Columns

In [7]:
schedule = schedule.rename(columns={'League':'VisitorLeague', 'Game':'VisitorGame', 'League.1':'HomeLeague', 'Game.1':'HomeGame'})
schedule.head()

,Date,Num,Day,Visitor,VisitorLeague,VisitorGame,Home,HomeLeague,HomeGame,Day/Night,Location,Postponed,Makeup
0,20250318,0,Tuesday,LAN,NL,1,CHN,NL,1,n,TOK01,NaN,NaN
1,20250319,0,Wednesday,LAN,NL,2,CHN,NL,2,n,TOK01,NaN,NaN
2,20250327,0,Thursday,MIL,NL,1,NYA,AL,1,d,NYC21,NaN,NaN
3,20250327,0,Thursday,BAL,AL,1,TOR,AL,1,d,TOR02,NaN,NaN
4,20250327,0,Thursday,BOS,AL,1,TEX,AL,1,d,ARL03,NaN,NaN


## 2. Renaming 'ATH' team names to 'OAK'

In [ ]:
preprocessor.rename_teams(['Home', 'Visitor'])

## 3. Adding a 'timestamp' column, which provides a `pd.Timestamp` for the game start.

In [9]:
def get_timestamp(date: int) -> pd.Timestamp:
    """For a given date int of format YYYYMMDD, gets its game start timestamp."""

    date = str(date)

    y = int(date[:4])
    m = int(date[4:6])
    d = int(date[6:])
    return pd.Timestamp(year=y, month=m, day=d)

# Add the timestamp column

schedule['timestamp'] = schedule['Date'].apply(get_timestamp)
schedule.head()

,Date,Num,Day,Visitor,VisitorLeague,VisitorGame,Home,HomeLeague,HomeGame,Day/Night,Location,Postponed,Makeup,timestamp
0,20250318,0,Tuesday,LAN,NL,1,CHN,NL,1,n,TOK01,NaN,NaN,2025-03-18
1,20250319,0,Wednesday,LAN,NL,2,CHN,NL,2,n,TOK01,NaN,NaN,2025-03-19
2,20250327,0,Thursday,MIL,NL,1,NYA,AL,1,d,NYC21,NaN,NaN,2025-03-27
3,20250327,0,Thursday,BAL,AL,1,TOR,AL,1,d,TOR02,NaN,NaN,2025-03-27
4,20250327,0,Thursday,BOS,AL,1,TEX,AL,1,d,ARL03,NaN,NaN,2025-03-27


## 4. Adding 'homedistancetraveled' and 'visdistancetraveled' Columns

In [ ]:
preprocessor.add_distance_traveled_cols('Home', 'Visitor', 'Location', over_seasons=False)

2430it [00:00, 31184.23it/s]


,Date,Num,Day,Visitor,VisitorLeague,VisitorGame,Home,HomeLeague,HomeGame,Day/Night,Location,Postponed,Makeup,timestamp,Latitude,Longitude,homedistancetraveled,visdistancetraveled
0,20250318,0,Tuesday,LAN,NL,1,CHN,NL,1,n,TOK01,NaN,NaN,2025-03-18,35.705526,139.751928,0.0,5473.620854
1,20250319,0,Wednesday,LAN,NL,2,CHN,NL,2,n,TOK01,NaN,NaN,2025-03-19,35.705526,139.751928,0.0,0.000000
2,20250327,0,Thursday,MIL,NL,1,NYA,AL,1,d,NYC21,NaN,NaN,2025-03-27,40.829586,-73.926413,0.0,736.818918
3,20250327,0,Thursday,BAL,AL,1,TOR,AL,1,d,TOR02,NaN,NaN,2025-03-27,43.641256,-79.389054,0.0,333.374579
4,20250327,0,Thursday,BOS,AL,1,TEX,AL,1,d,ARL03,NaN,NaN,2025-03-27,32.747361,-97.084167,0.0,1562.150215


## 5. Add Rest Day Columns

In [ ]:
preprocessor.add_rest_days_cols('Home', 'Visitor')

2430it [00:00, 9917.64it/s] 


,Date,Num,Day,Visitor,VisitorLeague,VisitorGame,Home,HomeLeague,HomeGame,Day/Night,Location,Postponed,Makeup,timestamp,Latitude,Longitude,homedistancetraveled,visdistancetraveled,homerestdays,visrestdays
0,20250318,0,Tuesday,LAN,NL,1,CHN,NL,1,n,TOK01,NaN,NaN,2025-03-18,35.705526,139.751928,0.0,5473.620854,NaN,NaN
1,20250319,0,Wednesday,LAN,NL,2,CHN,NL,2,n,TOK01,NaN,NaN,2025-03-19,35.705526,139.751928,0.0,0.000000,1.0,1.0
2,20250327,0,Thursday,MIL,NL,1,NYA,AL,1,d,NYC21,NaN,NaN,2025-03-27,40.829586,-73.926413,0.0,736.818918,NaN,NaN
3,20250327,0,Thursday,BAL,AL,1,TOR,AL,1,d,TOR02,NaN,NaN,2025-03-27,43.641256,-79.389054,0.0,333.374579,NaN,NaN
4,20250327,0,Thursday,BOS,AL,1,TEX,AL,1,d,ARL03,NaN,NaN,2025-03-27,32.747361,-97.084167,0.0,1562.150215,NaN,NaN


## 6. Save to `.csv`

In [16]:
schedule.to_csv(Path.cwd() / "analysis" / "data" / "clean" / f"{schedule_name}_clean.csv", index=False)